# Микробенчмарк скорости и памяти для деплоя на iPhone

Финальный ноутбук магистерской: насколько готовый pipeline практичен для
мобильного инференса. Меряем размер моделей на диске и в памяти, latency
end-to-end на CPU в одном потоке (прокси для Apple Neural Engine), на
CPU в нескольких потоках (общий CPU iPhone) и на GPU T4 (для контекста).
Также делаем ONNX-экспорт, чтобы дальше на macOS можно было сконвертировать
в CoreML и запустить на устройстве. Готовое число «X миллисекунд на блюдо»
и оценка размера приложения дают ответ, реализуемо ли решение в виде
обычного App Store-приложения или нужны компромиссы.

## 1. Подготовка окружения

Подключены три input-датасета: `nutrition5k-overhead-rgb-224` (тестовое
изображение и текст ингредиентов), `nutrition5k-visual-baseline`
(нормализация целей), `nutrition5k-conformal-intervals` (CQR-голова и
конформные поправки). Visual encoder и text encoder тянем с Hugging Face,
нужен Internet on на первом запуске.

In [1]:
import gc
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image

import matplotlib.pyplot as plt
import seaborn as sns

INPUT_DATA = Path("/kaggle/input/datasets/dbkarashev/nutrition5k-overhead-rgb-224/n5k_overhead")
INPUT_BASELINE = Path("/kaggle/input/datasets/dbkarashev/nutrition5k-visual-baseline/visual_baseline")
INPUT_CONFORMAL = Path("/kaggle/input/datasets/dbkarashev/nutrition5k-conformal-intervals/conformal_calibration")

WORK_DIR = Path("/kaggle/working/iphone_microbench")
ONNX_DIR = WORK_DIR / "onnx"
EDA_DIR = WORK_DIR / "eda"
for d in (WORK_DIR, ONNX_DIR, EDA_DIR):
    d.mkdir(parents=True, exist_ok=True)

RESULTS_FILE = WORK_DIR / "benchmark.json"
STATUS_FILE = WORK_DIR / "_status.json"

VIS_ENCODER = "facebook/dinov2-small"
TEXT_ENCODER = "sentence-transformers/all-MiniLM-L6-v2"
TARGETS = ["total_calories", "total_fat", "total_carb", "total_protein"]
QUANTILES = [0.05, 0.50, 0.95]

VIS_DIM = 384
TEXT_DIM = 384
HIDDEN = 256
DROPOUT = 0.2

WARMUP = 10
N_RUNS = 50


def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z")


def update_status(patch: dict) -> None:
    state = {}
    if STATUS_FILE.exists():
        try:
            state = json.loads(STATUS_FILE.read_text())
        except json.JSONDecodeError:
            pass
    state.update(patch)
    state["last_updated"] = now_iso()
    STATUS_FILE.write_text(json.dumps(state, indent=2, ensure_ascii=False))


print(f"GPU доступен: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  device: {torch.cuda.get_device_name(0)}")
print(f"Кол-во CPU: {os.cpu_count()}")

GPU доступен: True
  device: Tesla T4
Кол-во CPU: 4


## 2. Сборка inference pipeline

Полный путь от файла фото и строки ингредиентов до интервала с покрытием
90 %. Шаги: image → DINOv2 (CLS) → V; text → tokenizer → MiniLM → mean_pool
→ T; (V, T) → CQR-голова → три квантиля по четырем целям → денормализация
→ плюс конформная поправка из ноутбука 04 → пары `[lo, hi]`. Это и есть то,
что должно работать на устройстве.

In [2]:
from transformers import AutoImageProcessor, AutoModel, AutoTokenizer

vis_processor = AutoImageProcessor.from_pretrained(VIS_ENCODER)
vis_encoder = AutoModel.from_pretrained(VIS_ENCODER).eval()
text_tokenizer = AutoTokenizer.from_pretrained(TEXT_ENCODER)
text_encoder = AutoModel.from_pretrained(TEXT_ENCODER).eval()


def mlp(in_dim: int, hidden: int, out_dim: int, dropout: float) -> nn.Sequential:
    return nn.Sequential(
        nn.Linear(in_dim, hidden), nn.GELU(), nn.Dropout(dropout),
        nn.Linear(hidden, hidden), nn.GELU(), nn.Dropout(dropout),
        nn.Linear(hidden, out_dim),
    )


class GatedQuantile(nn.Module):
    def __init__(self, n_targets: int, n_quantiles: int):
        super().__init__()
        self.n_t = n_targets
        self.n_q = n_quantiles
        self.v_proj = nn.Linear(VIS_DIM, HIDDEN)
        self.t_proj = nn.Linear(TEXT_DIM, HIDDEN)
        self.gate = nn.Sequential(nn.Linear(2 * HIDDEN, HIDDEN), nn.Sigmoid())
        self.head = mlp(HIDDEN, HIDDEN, n_targets * n_quantiles, DROPOUT)

    def forward(self, v, t):
        v_h = self.v_proj(v)
        t_h = self.t_proj(t)
        g = self.gate(torch.cat([v_h, t_h], dim=-1))
        fused = g * v_h + (1.0 - g) * t_h
        out = self.head(fused).view(-1, self.n_t, self.n_q)
        return torch.sort(out, dim=-1).values


cqr_head = GatedQuantile(len(TARGETS), len(QUANTILES))
cqr_head.load_state_dict(torch.load(INPUT_CONFORMAL / "models" / "cqr_head.pt", map_location="cpu"))
cqr_head.eval()

norm = json.loads((INPUT_BASELINE / "target_norm.json").read_text())
mean = torch.tensor(norm["mean"], dtype=torch.float32)
std = torch.tensor(norm["std"], dtype=torch.float32)

conformal_q = json.loads((INPUT_CONFORMAL / "conformal_quantiles.json").read_text())["cqr_q"]
Q_correction = torch.tensor([conformal_q[t] for t in TARGETS], dtype=torch.float32)


def mean_pool(token_emb: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).float()
    return (token_emb * mask).sum(1) / mask.sum(1).clamp(min=1e-9)


def predict_interval(image_path: Path, ingredients_text: str, device: torch.device) -> dict:
    img = Image.open(image_path).convert("RGB")
    pixels = vis_processor(images=img, return_tensors="pt")["pixel_values"].to(device)
    tokens = text_tokenizer(ingredients_text, padding=True, truncation=True, max_length=128, return_tensors="pt")
    tokens = {k: v.to(device) for k, v in tokens.items()}

    with torch.inference_mode():
        v_out = vis_encoder(pixel_values=pixels)
        V = v_out.last_hidden_state[:, 0, :]
        t_out = text_encoder(**tokens)
        T = mean_pool(t_out.last_hidden_state, tokens["attention_mask"])
        q = cqr_head(V, T)

    q_real = q.cpu() * std.view(1, -1, 1) + mean.view(1, -1, 1)
    lo = q_real[0, :, 0] - Q_correction
    hi = q_real[0, :, 2] + Q_correction
    return {t: (float(lo[i]), float(hi[i])) for i, t in enumerate(TARGETS)}


test_ids = [l.strip() for l in (INPUT_DATA / "splits" / "test.txt").read_text().splitlines() if l.strip()]
sample_id = test_ids[0]
sample_img = INPUT_DATA / "images_224" / f"{sample_id}.jpg"
dishes_df = pd.read_parquet(INPUT_DATA / "metadata" / "dishes.parquet")
sample_text = dishes_df.set_index("dish_id").loc[sample_id, "ingredient_names_str"]

print(f"Образец: {sample_id}")
print(f"  ингредиенты: {sample_text[:120]}{'...' if len(sample_text) > 120 else ''}")
print(f"\nИнтервалы (90 %, CPU sanity check):")
intervals = predict_interval(sample_img, sample_text, torch.device("cpu"))
for t, (lo, hi) in intervals.items():
    print(f"  {t:<18} [{lo:7.2f}, {hi:7.2f}]")

preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

The image processor of type `BitImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Образец: dish_1556575327
  ингредиенты: celery, olives, brussels sprouts

Интервалы (90 %, CPU sanity check):
  total_calories     [   2.22,  112.56]
  total_fat          [  -0.81,    5.30]
  total_carb         [   1.47,   11.41]
  total_protein      [  -0.21,    6.91]


## 3. Размер моделей на диске и в памяти

Раздельно для всех компонентов: visual encoder, text encoder и две головы.
Считаем сразу три цифры — в FP32 (как обучено), FP16 (типичная квантизация
для CoreML без потери качества) и INT8 (агрессивная квантизация ANE).

In [3]:
def n_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


components = {
    "visual_dinov2_small": vis_encoder,
    "text_minilm_l6_v2": text_encoder,
    "cqr_head_gated": cqr_head,
}

sizes_mb = {}
for name, model in components.items():
    n = n_params(model)
    sizes_mb[name] = {
        "params": n,
        "fp32_mb": round(n * 4 / 1024**2, 2),
        "fp16_mb": round(n * 2 / 1024**2, 2),
        "int8_mb": round(n * 1 / 1024**2, 2),
    }

total = {
    "params": sum(s["params"] for s in sizes_mb.values()),
    "fp32_mb": round(sum(s["fp32_mb"] for s in sizes_mb.values()), 2),
    "fp16_mb": round(sum(s["fp16_mb"] for s in sizes_mb.values()), 2),
    "int8_mb": round(sum(s["int8_mb"] for s in sizes_mb.values()), 2),
}

print(f"  {'компонент':<24}{'params':>12}{'FP32 МБ':>10}{'FP16 МБ':>10}{'INT8 МБ':>10}")
for name, s in sizes_mb.items():
    print(f"  {name:<24}{s['params']:>12,d}{s['fp32_mb']:>10.2f}{s['fp16_mb']:>10.2f}{s['int8_mb']:>10.2f}")
print(f"  {'итог':<24}{total['params']:>12,d}{total['fp32_mb']:>10.2f}{total['fp16_mb']:>10.2f}{total['int8_mb']:>10.2f}")

  компонент                     params   FP32 МБ   FP16 МБ   INT8 МБ
  visual_dinov2_small       22,056,576     84.14     42.07     21.03
  text_minilm_l6_v2         22,713,216     86.64     43.32     21.66
  cqr_head_gated               463,116      1.77      0.88      0.44
  итог                      45,232,908    172.55     86.27     43.13


## 4. Latency end-to-end в трех режимах

Меряем три сценария: GPU T4 как верхняя граница, CPU в одном потоке
(наиболее близко к работе через Apple Neural Engine — ANE тоже работает
в одном вычислительном блоке) и CPU во всех потоках (нагрузка на
обычные CPU-ядра iPhone). Перед каждой серией прогон warmup-итераций,
дальше N_RUNS замеров.

In [4]:
def benchmark(device: torch.device, threads: int | None) -> dict:
    if threads is not None:
        torch.set_num_threads(threads)

    vis_encoder.to(device)
    text_encoder.to(device)
    cqr_head.to(device)

    for _ in range(WARMUP):
        predict_interval(sample_img, sample_text, device)
    if device.type == "cuda":
        torch.cuda.synchronize()

    times = []
    for _ in range(N_RUNS):
        t0 = time.perf_counter()
        predict_interval(sample_img, sample_text, device)
        if device.type == "cuda":
            torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)

    return {
        "mean_ms": round(float(np.mean(times)), 2),
        "median_ms": round(float(np.median(times)), 2),
        "p95_ms": round(float(np.percentile(times, 95)), 2),
        "min_ms": round(float(np.min(times)), 2),
        "max_ms": round(float(np.max(times)), 2),
    }


latency = {}

if torch.cuda.is_available():
    print("[bench] GPU T4")
    latency["gpu_t4"] = benchmark(torch.device("cuda"), threads=None)
    print(f"  {latency['gpu_t4']}")

print("[bench] CPU 1 поток (прокси для Apple Neural Engine)")
latency["cpu_1thread"] = benchmark(torch.device("cpu"), threads=1)
print(f"  {latency['cpu_1thread']}")

print(f"[bench] CPU {os.cpu_count()} потоков (общий CPU iPhone)")
latency[f"cpu_{os.cpu_count()}threads"] = benchmark(torch.device("cpu"), threads=os.cpu_count())
print(f"  {latency[f'cpu_{os.cpu_count()}threads']}")

torch.set_num_threads(os.cpu_count())
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

[bench] GPU T4
  {'mean_ms': 16.57, 'median_ms': 15.83, 'p95_ms': 18.43, 'min_ms': 15.01, 'max_ms': 23.94}
[bench] CPU 1 поток (прокси для Apple Neural Engine)
  {'mean_ms': 167.3, 'median_ms': 165.94, 'p95_ms': 176.22, 'min_ms': 161.89, 'max_ms': 189.47}
[bench] CPU 4 потоков (общий CPU iPhone)
  {'mean_ms': 109.37, 'median_ms': 106.23, 'p95_ms': 119.03, 'min_ms': 103.05, 'max_ms': 196.04}


## 5. Пиковая память

На GPU — `torch.cuda.max_memory_allocated`. На CPU — RSS процесса
до и после прогона.

In [5]:
import psutil

memory = {}
process = psutil.Process()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    vis_encoder.to("cuda"); text_encoder.to("cuda"); cqr_head.to("cuda")
    predict_interval(sample_img, sample_text, torch.device("cuda"))
    torch.cuda.synchronize()
    memory["gpu_peak_mb"] = round(torch.cuda.max_memory_allocated() / 1024**2, 2)

vis_encoder.to("cpu"); text_encoder.to("cpu"); cqr_head.to("cpu")
gc.collect()
rss_before = process.memory_info().rss
predict_interval(sample_img, sample_text, torch.device("cpu"))
rss_after = process.memory_info().rss
memory["cpu_rss_after_mb"] = round(rss_after / 1024**2, 2)
memory["cpu_rss_delta_mb"] = round((rss_after - rss_before) / 1024**2, 2)

print("Пиковая память:")
for k, v in memory.items():
    print(f"  {k:<24}{v:>10.2f} МБ")

Пиковая память:
  gpu_peak_mb                 189.39 МБ
  cpu_rss_after_mb           1719.25 МБ
  cpu_rss_delta_mb              0.00 МБ


## 6. ONNX-экспорт для CoreML-конвертации на macOS

Экспортируем три компонента в ONNX. Дальше на Mac:
`pip install coremltools` и стандартный `ct.convert(onnx_path, ...)` с
`compute_precision=ct.precision.FLOAT16` дает CoreML `.mlpackage`,
который запускается на ANE с минимальной потерей качества. Tokenizer
и image preprocessing на устройстве — стандартные CoreML/Vision API.

In [6]:
torch.set_num_threads(1)
vis_encoder.eval(); text_encoder.eval(); cqr_head.eval()

dummy_pixels = torch.randn(1, 3, 224, 224)
dummy_input_ids = torch.zeros((1, 32), dtype=torch.long)
dummy_attention = torch.ones((1, 32), dtype=torch.long)
dummy_v = torch.randn(1, VIS_DIM)
dummy_t = torch.randn(1, TEXT_DIM)


class VisualWrapper(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder

    def forward(self, pixel_values):
        return self.encoder(pixel_values=pixel_values).last_hidden_state[:, 0, :]


class TextWrapper(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        return (out * mask).sum(1) / mask.sum(1).clamp(min=1e-9)


export_targets = [
    ("visual_dinov2_small.onnx", VisualWrapper(vis_encoder), (dummy_pixels,),
     ["pixel_values"], ["embedding"]),
    ("text_minilm_l6_v2.onnx", TextWrapper(text_encoder), (dummy_input_ids, dummy_attention),
     ["input_ids", "attention_mask"], ["embedding"]),
    ("cqr_head.onnx", cqr_head, (dummy_v, dummy_t),
     ["v", "t"], ["quantiles"]),
]

# dynamo=False явно выбирает legacy tracing-based exporter, который не
# требует пакета onnxscript. Новый dynamo-based путь в свежем torch стал
# default, но onnxscript в Kaggle-образе не установлен.
for filename, model, args, in_names, out_names in export_targets:
    out_path = ONNX_DIR / filename
    torch.onnx.export(
        model, args, str(out_path),
        input_names=in_names, output_names=out_names,
        dynamic_axes={n: {0: "batch"} for n in (in_names + out_names)},
        opset_version=14, do_constant_folding=True, dynamo=False,
    )
    print(f"  {filename}: {out_path.stat().st_size / 1024**2:.2f} МБ")

update_status({"step_6_onnx_exported": True})

/tmp/ipykernel_23/169900158.py:45: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/usr/local/lib/python3.12/dist-packages/transformers/models/dinov2/modeling_dinov2.py:142: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if num_channels != self.num_channels:
/usr/local/lib/python3.12/dist-packages/transformers/integrations/sdpa_attention.py:77: TracerWarning: Converting a tensor to a Python boolean might cause the trace to 

  visual_dinov2_small.onnx: 84.37 МБ


/usr/local/lib/python3.12/dist-packages/transformers/masking_utils.py:171: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if (padding_length := kv_length + kv_offset - attention_mask.shape[-1]) > 0:
/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset11.py:954: UserWarning: Exporting aten::index operator of advanced indexing in opset 14 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)


  text_minilm_l6_v2.onnx: 86.23 МБ
  cqr_head.onnx: 1.77 МБ


## 7. Графики и сводный отчет

Латенси-бар-чарт по режимам и pie размера модели по компонентам.
В summary включены оценки для iPhone: ANE-производительность типичной
ViT-Small в FP16 — порядка 5-10 мс на инференс энкодера, что соответствует
`cpu_1thread / 5..10`. Это грубая, но рабочая прокси.

In [7]:
sns.set_theme(style="whitegrid", context="talk")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
labels = list(latency.keys())
mean_ms = [latency[k]["mean_ms"] for k in labels]
p95_ms = [latency[k]["p95_ms"] for k in labels]
x = np.arange(len(labels))
axes[0].bar(x - 0.2, mean_ms, 0.4, label="mean", color="steelblue")
axes[0].bar(x + 0.2, p95_ms, 0.4, label="p95", color="darkorange")
axes[0].set(xticks=x, ylabel="мс на блюдо", title="Latency end-to-end")
axes[0].set_xticklabels(labels, rotation=15)
axes[0].legend()

names = list(sizes_mb.keys())
fp16_vals = [sizes_mb[n]["fp16_mb"] for n in names]
axes[1].pie(fp16_vals, labels=names, autopct="%1.1f%%", startangle=90,
            colors=["steelblue", "seagreen", "darkorange"])
axes[1].set(title=f"Размер компонентов в FP16 (итог {total['fp16_mb']:.1f} МБ)")
fig.tight_layout()
fig.savefig(EDA_DIR / "benchmark_summary.png", dpi=120)
plt.close(fig)

print(f"  графики в {EDA_DIR}")

  графики в /kaggle/working/iphone_microbench/eda


## 8. Финальный лог

Сохраняем все замеры в `benchmark.json` и обновляем `_status.json`.
В выводе — оценка реализуемости приложения на iPhone и где главный
трейдоф.

In [8]:
results = {
    "models": {
        "visual_encoder": VIS_ENCODER,
        "text_encoder": TEXT_ENCODER,
        "cqr_head": "GatedQuantile (DINOv2-small + MiniLM + 4 цели x 3 квантиля)",
    },
    "sizes_mb": {**sizes_mb, "total": total},
    "latency_ms": latency,
    "memory_mb": memory,
    "iphone_estimate": {
        "ane_optimistic_ms": round(latency["cpu_1thread"]["mean_ms"] / 10, 1),
        "ane_pessimistic_ms": round(latency["cpu_1thread"]["mean_ms"] / 5, 1),
        "comment": "ANE дает порядка 5-10x speedup относительно single-thread CPU"
                   " для типичной ViT-Small FP16. Реальные числа зависят от модели"
                   " iPhone и того, удастся ли весь граф уложить на ANE.",
    },
    "onnx_files": [
        {"file": p.name, "size_mb": round(p.stat().st_size / 1024**2, 2)}
        for p in sorted(ONNX_DIR.glob("*.onnx"))
    ],
    "config": {"warmup": WARMUP, "n_runs": N_RUNS},
    "timestamp": now_iso(),
}
RESULTS_FILE.write_text(json.dumps(results, indent=2, ensure_ascii=False))
update_status({"step_8_summary_written": True, "summary": results})

print(json.dumps(results, indent=2, ensure_ascii=False))
print()
print("Дальше на macOS:")
print("  pip install coremltools onnx")
print("  python -c \"import coremltools as ct; m = ct.convert('visual_dinov2_small.onnx',"
      " compute_precision=ct.precision.FLOAT16); m.save('visual.mlpackage')\"")
print("Аналогично для остальных onnx-файлов. Это даст пакет для запуска на ANE.")

{
  "models": {
    "visual_encoder": "facebook/dinov2-small",
    "text_encoder": "sentence-transformers/all-MiniLM-L6-v2",
    "cqr_head": "GatedQuantile (DINOv2-small + MiniLM + 4 цели x 3 квантиля)"
  },
  "sizes_mb": {
    "visual_dinov2_small": {
      "params": 22056576,
      "fp32_mb": 84.14,
      "fp16_mb": 42.07,
      "int8_mb": 21.03
    },
    "text_minilm_l6_v2": {
      "params": 22713216,
      "fp32_mb": 86.64,
      "fp16_mb": 43.32,
      "int8_mb": 21.66
    },
    "cqr_head_gated": {
      "params": 463116,
      "fp32_mb": 1.77,
      "fp16_mb": 0.88,
      "int8_mb": 0.44
    },
    "total": {
      "params": 45232908,
      "fp32_mb": 172.55,
      "fp16_mb": 86.27,
      "int8_mb": 43.13
    }
  },
  "latency_ms": {
    "gpu_t4": {
      "mean_ms": 16.57,
      "median_ms": 15.83,
      "p95_ms": 18.43,
      "min_ms": 15.01,
      "max_ms": 23.94
    },
    "cpu_1thread": {
      "mean_ms": 167.3,
      "median_ms": 165.94,
      "p95_ms": 176.22,
      "min